In [ ]:
import math
import os
import pandas as pd
import numpy as np
import json 
import random
import lib.AttackLib_copy as al

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
# site = 'dataset10'
site = 'dataset1'
# orignal_csv = "training_20252810_18:32:01"
orignal_csv = "training_20253010_12:24:37"
df = pd.read_csv('./'+site+'/training/'+orignal_csv+'.csv')
np.where(df['Id'].isnull())

(array([], dtype=int64),)

In [29]:
from datetime import datetime
now = datetime.now()
time = now.strftime("%Y%m%d_%H:%M:%S")

# Save labelled ground truth 

In [30]:
# create benign csv
new_df = df.copy()
new_df['attacked'] = np.where(new_df['Id'].isnull(),1,0)
new_df.head(1)

,Id,timestamp,positionX,positionY,positionZ,sizeX,sizeY,sizeZ,VelX,VelY,VelZ,Vel,Class,rot_x,rot_y,edge,ts,departure,destination,attacked
0,220,1.607521e+12,5.418094,2.576576,-0.589794,3.298336,1.418282,5.11065,-5.511428,1.593727,0.095754,20.656905,VEHICLE,4.464588,-4.007735,1151278800#2,0,1151278800#2,1090414507,0


In [31]:
# save benign csv
new_df.to_csv('./'+site+'/test/benign/'+time+'_benign'+'.csv',mode='x',index=False)

# Inject attack

In [32]:
new_df = df.copy()
with open('./config/attack_config.json','r') as f:
    attack_cfg = json.load(f)

with open(attack_cfg['template'],'r') as f:
    attack_template = json.load(f)
attack_cfg, attack_template

({'type': 'random_poistion_offset',
  'attack_rate': 0.2,
  'template': './config/template/random_position_offset.json',
  'duration': 50},
 {'insert': 'random',
  'x_min': 0.0,
  'x_max': 3.0,
  'y_min': 0.0,
  'y_max': 3.0,
  'vx_min': 0.0,
  'vx_max': 0.0,
  'vy_min': 0.0,
  'vy_max': 0.0})

In [33]:
cfg_params=pd.concat([pd.DataFrame.from_dict(attack_cfg, orient='index'),pd.DataFrame.from_dict(attack_template, orient='index')])
cfg_params.to_csv('./'+site+'/test/attacked/'+time+'_param'+'.log',sep=':', mode='x',header=False)

In [34]:
al.attack_types[attack_cfg['type']]

1

In [35]:
attacked_df  = al.AttackModel(new_df, attack_cfg, attack_template).inject_attack()

In [36]:
attacked_df.to_csv('./'+site+'/test/attacked/'+time+'_'+attack_cfg['type']+'_pr'+str(attack_cfg['attack_rate'])+'.csv',index=False)